# FHIR Patient Data Quality Profiling

## Purpose

In this notebook, I profile the Silver FHIR Patient dataset and define the
data-quality rules that will later be enforced directly by the Lakeflow
Bronze-to-Silver transformation.

I am not creating another cleaned Patient table in this notebook.

The purpose of this stage is to:

- identify incomplete or invalid Patient records
- measure current quality-rule violations
- distinguish critical identifiers from optional FHIR attributes
- classify rules as warning, drop, or fail
- prepare reusable Lakeflow expectation definitions

### Source

`health_insurance.silver.fhir_patient`

### Production design

The final production flow will be:

Bronze  
↓  
FHIR Patient transformation  
+  
Lakeflow expectations  
↓  
Validated Silver Patient  
↓  
Gold

### Data-quality approach

FHIR contains many optional attributes. I therefore do not treat every missing
value as invalid data.

I distinguish between:

- attributes required for a usable Patient entity
- values that should satisfy logical constraints
- optional attributes whose absence should only be monitored



In [0]:
# loading the current Silver Patient dataset for quality profiling.

from pyspark.sql import functions as F

PATIENT_TABLE = "health_insurance.silver.fhir_patient"

patient_df = spark.table(PATIENT_TABLE)

print(f"Patient rows: {patient_df.count():,}")

patient_df.printSchema()

display(patient_df.limit(10))

In [0]:
# defining candidate Patient quality rules by severity.

PATIENT_WARN_RULES = {
    "medical_record_number_present":
        "medical_record_number IS NOT NULL",

    "given_name_present":
        "given_name IS NOT NULL",

    "family_name_present":
        "family_name IS NOT NULL",

    "recognized_gender":
        "gender IN ('MALE', 'FEMALE', 'OTHER', 'UNKNOWN')",

    "phone_present":
        "phone IS NOT NULL"
}


PATIENT_DROP_RULES = {
    "patient_id_present":
        "patient_id IS NOT NULL",

    "birth_date_present":
        "birth_date IS NOT NULL"
}


PATIENT_FAIL_RULES = {
    "birth_date_not_future":
        "birth_date <= current_date()",

    "age_logically_valid":
        "age BETWEEN 0 AND 120"
}

In [0]:
# measuring how many Patient rows violate each candidate quality rule.

def profile_rules(df, rules, severity):

    results = []

    total_rows = df.count()

    for rule_name, condition in rules.items():

        failed_rows = (
            df
            .filter(
                f"NOT ({condition}) OR ({condition}) IS NULL"
            )
            .count()
        )

        results.append(
            (
                rule_name,
                severity,
                condition,
                total_rows,
                failed_rows,
                round(
                    failed_rows / total_rows * 100,
                    2
                ) if total_rows else 0.0
            )
        )

    return results

In [0]:
# profiling all proposed Patient quality rules.

patient_quality_results = []

patient_quality_results += profile_rules(
    patient_df,
    PATIENT_WARN_RULES,
    "WARN"
)

patient_quality_results += profile_rules(
    patient_df,
    PATIENT_DROP_RULES,
    "DROP"
)

patient_quality_results += profile_rules(
    patient_df,
    PATIENT_FAIL_RULES,
    "FAIL"
)

In [0]:
# presenting the Patient quality profile as a structured result.

patient_quality_profile_df = spark.createDataFrame(
    patient_quality_results,
    [
        "rule_name",
        "severity",
        "constraint",
        "total_rows",
        "failed_rows",
        "failed_percentage"
    ]
)

display(
    patient_quality_profile_df
    .orderBy(
        "severity",
        F.desc("failed_percentage")
    )
)

In [0]:
# profiling the completeness of optional Patient attributes
# before deciding their final quality severity.

patient_df.select(
    F.count("*").alias("total_patients"),

    F.sum(
        F.col("medical_record_number").isNull().cast("int")
    ).alias("missing_mrn"),

    F.sum(
        F.col("given_name").isNull().cast("int")
    ).alias("missing_given_name"),

    F.sum(
        F.col("family_name").isNull().cast("int")
    ).alias("missing_family_name"),

    F.sum(
        F.col("phone").isNull().cast("int")
    ).alias("missing_phone"),

    F.sum(
        F.col("city").isNull().cast("int")
    ).alias("missing_city"),

    F.sum(
        F.col("country").isNull().cast("int")
    ).alias("missing_country")
).show()

In [0]:
# defining the finalized Patient quality contract.

PATIENT_WARN_RULES = {
    "birth_date_present":
        "birth_date IS NOT NULL",

    "medical_record_number_present":
        "medical_record_number IS NOT NULL",

    "given_name_present":
        "given_name IS NOT NULL",

    "family_name_present":
        "family_name IS NOT NULL",

    "recognized_gender":
        "gender IN ('MALE', 'FEMALE', 'OTHER', 'UNKNOWN')",

    "phone_present":
        "phone IS NOT NULL"
}


PATIENT_DROP_RULES = {
    "patient_id_present":
        "patient_id IS NOT NULL",

    "age_logically_valid":
        "age IS NULL OR age BETWEEN 0 AND 120",

    "birth_date_not_future":
        "birth_date IS NULL OR birth_date <= current_date()"
}


PATIENT_FAIL_RULES = {}

In [0]:
# preparing the finalized Patient rules for storage
# in the reusable quality_rules.py module.

patient_quality_code = '''
# === PATIENT QUALITY RULES START ===

PATIENT_WARN_RULES = {
    "birth_date_present":
        "birth_date IS NOT NULL",

    "medical_record_number_present":
        "medical_record_number IS NOT NULL",

    "given_name_present":
        "given_name IS NOT NULL",

    "family_name_present":
        "family_name IS NOT NULL",

    "recognized_gender":
        "gender IN ('MALE', 'FEMALE', 'OTHER', 'UNKNOWN')",

    "phone_present":
        "phone IS NOT NULL"
}


PATIENT_DROP_RULES = {
    "patient_id_present":
        "patient_id IS NOT NULL",

    "age_logically_valid":
        "age IS NULL OR age BETWEEN 0 AND 120",

    "birth_date_not_future":
        "birth_date IS NULL OR birth_date <= current_date()"
}


PATIENT_FAIL_RULES = {}

# === PATIENT QUALITY RULES END ===
'''

In [0]:
# locating the reusable quality-rules module
# inside my Databricks Git repository.

QUALITY_RULES_PATH = (
    "/Workspace/Khaoula healthy insurance project/"
    "Khaoula-healthy-insuarance-project/"
    "04-data-quality/"
    "quality_rules.py"
)

In [0]:
# adding or updating the Patient section
# without overwriting the existing Claims rules.

from pathlib import Path
import re

quality_file = Path(QUALITY_RULES_PATH)

existing_code = quality_file.read_text(
    encoding="utf-8"
)

start_marker = "# === PATIENT QUALITY RULES START ==="
end_marker = "# === PATIENT QUALITY RULES END ==="

pattern = (
    re.escape(start_marker)
    + r".*?"
    + re.escape(end_marker)
)

if start_marker in existing_code:
    updated_code = re.sub(
        pattern,
        patient_quality_code.strip(),
        existing_code,
        flags=re.DOTALL
    )
else:
    updated_code = (
        existing_code.rstrip()
        + "\n\n\n"
        + patient_quality_code.strip()
        + "\n"
    )

quality_file.write_text(
    updated_code,
    encoding="utf-8"
)

print("Patient quality rules saved successfully.")
print(QUALITY_RULES_PATH)

In [0]:
# verifying that the Patient quality contract
# is present in my reusable rules module.

saved_code = quality_file.read_text(
    encoding="utf-8"
)

print(saved_code)